In [1]:
!pip install openpyxl
import pandas as pd
import re

In [3]:
# ================= IEEE =================
def parse_ieee(text):
    articles = text.split("\n\n")
    data = []

    for article in articles:
        if not article.strip():
            continue

        lines = article.strip().split("\n")
        main_line = lines[0]

        authors = main_line.split('",')[0]

        title_match = re.search(r'"(.*?)"', main_line)
        title = title_match.group(1) if title_match else ""

        journal_match = re.search(r'in (.*?), vol', main_line)
        journal = journal_match.group(1) if journal_match else ""

        year_match = re.search(r', (\d{4}),', main_line)
        year = year_match.group(1) if year_match else ""

        doi_match = re.search(r'doi: ([^\.\n]+)', main_line)
        doi = doi_match.group(1) if doi_match else ""

        keywords = ""
        if len(lines) > 1:
            kw_match = re.search(r'keywords: {(.*?)}', lines[1])
            if kw_match:
                keywords = kw_match.group(1)

        data.append({
            "Title": title,
            "Authors": authors,
            "Year": year,
            "Journal": journal,
            "Abstract": "",
            "DOI": doi,
            "Keywords": keywords,
            "Source_DB": "IEEE"
        })

    return pd.DataFrame(data)


# ================= GOOGLE SCHOLAR =================
def parse_google_scholar(text):
    articles = text.split("------------------------------------------------------------")
    data = []

    for article in articles:
        if not article.strip():
            continue

        title = re.search(r'Title: (.*)', article)
        authors = re.search(r'Authors: (.*)', article)
        year = re.search(r'Year: (\d{4})', article)
        source = re.search(r'Source: (.*)', article)
        summary = re.search(r'Summary: (.*)', article)

        data.append({
            "Title": title.group(1) if title else "",
            "Authors": authors.group(1) if authors else "",
            "Year": year.group(1) if year else "",
            "Journal": source.group(1) if source else "",
            "Abstract": summary.group(1) if summary else "",
            "DOI": "",
            "Keywords": "",
            "Source_DB": "Google Scholar"
        })

    return pd.DataFrame(data)


# ================= SCOPUS =================
def parse_scopus(text):
    articles = re.split(r'\n(?=[A-Z][a-zA-Z\-]+.*,)', text)
    data = []

    for article in articles:
        if "DOI:" not in article:
            continue

        lines = article.strip().split("\n")
        authors = lines[0].strip()
        title = lines[2].strip() if len(lines) > 2 else ""

        year = ""
        journal = ""
        year_journal_match = re.search(r'\((\d{4})\)\s*(.*)', article)
        if year_journal_match:
            year = year_journal_match.group(1)
            journal = year_journal_match.group(2).split(",")[0]

        doi_match = re.search(r'DOI:\s*(\S+)', article)
        doi = doi_match.group(1) if doi_match else ""

        abstract_match = re.search(r'ABSTRACT:\s*(.*?)(?:AUTHOR KEYWORDS:|INDEX KEYWORDS:)', article, re.S)
        abstract = abstract_match.group(1).strip() if abstract_match else ""

        keywords_match = re.search(r'AUTHOR KEYWORDS:\s*(.*)', article)
        keywords = keywords_match.group(1).strip() if keywords_match else ""

        data.append({
            "Title": title,
            "Authors": authors,
            "Year": year,
            "Journal": journal,
            "Abstract": abstract,
            "DOI": doi,
            "Keywords": keywords,
            "Source_DB": "Scopus"
        })

    return pd.DataFrame(data)

In [4]:
with open("scopus_export_Mar 31-2026_fc3be87f-2e9d-4ea0-8c22-2e3d855f3cec.txt", "r", encoding="utf-8") as f:
    scopus_text = f.read()

with open("IEEE Xplore Citation Plain Text Download 2026.3.31.16.1.40.txt", "r", encoding="utf-8") as f:
    ieee_text = f.read()

with open("google_scholar.txt", "r", encoding="utf-8") as f:
    gs_text = f.read()
    

In [6]:
df_scopus = parse_scopus(scopus_text)
df_ieee = parse_ieee(ieee_text)
df_gs = parse_google_scholar(gs_text)

df_all = pd.concat([df_scopus, df_ieee, df_gs], ignore_index=True)

In [13]:
!pip install openpyxl
def normalize_doi(doi):
    if not doi:
        return ""
    return doi.lower().replace("https://doi.org/", "").strip()

# Nettoyage titres
df_all["Title_clean"] = df_all["Title"].str.lower().str.strip()

# Nettoyage DOI
df_all["DOI"] = df_all["DOI"].fillna("")
df_all["DOI"] = df_all["DOI"].apply(normalize_doi)

# 1. Déduplication par DOI (si DOI existe)
df_doi = df_all[df_all["DOI"] != ""].drop_duplicates(subset="DOI")

# 2. Déduplication par titre (pour ceux sans DOI)
df_no_doi = df_all[df_all["DOI"] == ""].drop_duplicates(subset="Title_clean")

# Fusion finale
df_final = pd.concat([df_doi, df_no_doi], ignore_index=True)

print("Avant:", len(df_all))
print("Après:", len(df_final))

# Export
df_final.to_excel("visuo_haptic_multimodal_review1_2026.xlsx", index=False)



Avant: 243
Après: 219
